# Q4: Load & Analyze Oxford 102 Flowers Dataset
### Understand dataset structure, statistics, and visualize image-text pairs
**Pipeline:** Download → Load → Analyze Statistics → Visualize → Gradio UI

In [ ]:
# CELL 1: Install Dependencies
!pip install gradio torch torchvision matplotlib numpy Pillow scipy -q
print('Done!')

In [ ]:
# CELL 2: Imports
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import Flowers102
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import gradio as gr
import os
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = './flowers_data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print('Imports done!')

In [ ]:
# CELL 3: Download Oxford 102 Flowers Dataset

print('Downloading Oxford 102 Flowers dataset (~330MB)...')
print('This may take 2-3 minutes on Colab...')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


train_dataset = Flowers102(root=DATA_DIR, split='train', transform=transform, download=True)
val_dataset   = Flowers102(root=DATA_DIR, split='val',   transform=transform, download=True)
test_dataset  = Flowers102(root=DATA_DIR, split='test',  transform=transform, download=True)

print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Test samples  : {len(test_dataset)}')
print(f'Total samples : {len(train_dataset) + len(val_dataset) + len(test_dataset)}')
print(f'Num classes   : {len(train_dataset.classes) if hasattr(train_dataset, "classes") else 102}')
print('Dataset downloaded!')

In [ ]:
# CELL 4: Define flower class names (102 classes)
FLOWER_NAMES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',
    'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood',
    'globe thistle', 'snapdragon', 'colt s foot', 'king protea', 'spear thistle',
    'yellow iris', 'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower',
    'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary', 'red ginger',
    'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian',
    'artichoke', 'sweet william', 'carnation', 'garden phlox', 'love in the mist',
    'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower',
    'great masterwort', 'siam tulip', 'lenten rose', 'barberton daisy', 'daffodil',
    'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold',
    'buttercup', 'oxeye daisy', 'common dandelion', 'petunia', 'wild pansy',
    'primula', 'sunflower', 'pelargonium', 'bishop of llandaff', 'gaura', 'geranium',
    'orange dahlia', 'pink-yellow dahlia', 'cautleya spicata', 'japanese anemone',
    'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum', 'spring crocus',
    'bearded iris', 'windflower', 'tree poppy', 'gazania', 'azalea', 'water lily',
    'rose', 'thorn apple', 'morning glory', 'passion flower', 'lotus', 'toad lily',
    'anthurium', 'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose',
    'tree mallow', 'magnolia', 'cyclamen', 'watercress', 'canna lily', 'hippeastrum',
    'bee balm', 'ball moss', 'foxglove', 'bougainvillea', 'camellia', 'mallow',
    'mexican petunia', 'bromelia', 'blanket flower', 'trumpet creeper', 'blackberry lily'
]

print(f'Total flower classes defined: {len(FLOWER_NAMES)}')
print('First 10 classes:')
for i, name in enumerate(FLOWER_NAMES[:10]):
    print(f'  Class {i:3d}: {name}')

In [ ]:
# CELL 5: Generate synthetic text descriptions


import random
random.seed(42)

COLOR_WORDS   = ['vibrant', 'delicate', 'rich', 'pale', 'bright', 'deep', 'soft', 'vivid']
PETAL_WORDS   = ['rounded', 'pointed', 'layered', 'ruffled', 'slender', 'broad', 'curved']
TEXTURE_WORDS = ['silky', 'velvety', 'waxy', 'papery', 'smooth', 'textured', 'glossy']
SHAPE_WORDS   = ['trumpet-shaped', 'star-shaped', 'cup-shaped', 'bell-shaped', 'flat', 'dome-shaped']

def generate_description(flower_name, label_idx):
    templates = [
        f'A {random.choice(COLOR_WORDS)} {flower_name} with {random.choice(PETAL_WORDS)} petals and {random.choice(TEXTURE_WORDS)} texture.',
        f'This {flower_name} displays {random.choice(SHAPE_WORDS)} blooms with {random.choice(TEXTURE_WORDS)} {random.choice(PETAL_WORDS)} petals.',
        f'A beautiful {flower_name} featuring {random.choice(COLOR_WORDS)} {random.choice(PETAL_WORDS)} petals arranged in a {random.choice(SHAPE_WORDS)} form.',
        f'The {flower_name} has {random.choice(TEXTURE_WORDS)} petals that are {random.choice(PETAL_WORDS)} and {random.choice(COLOR_WORDS)} in appearance.',
    ]
    return random.choice(templates)


CLASS_DESCRIPTIONS = {
    i: generate_description(name, i) for i, name in enumerate(FLOWER_NAMES)
}

print('Sample text descriptions:')
for i in [0, 10, 50, 75, 101]:
    print(f'  Class {i:3d} ({FLOWER_NAMES[i]}):')
    print(f'    "{CLASS_DESCRIPTIONS[i]}"')
    print()

In [ ]:
# CELL 6: Dataset Statistics Analysis
print('Analyzing dataset statistics...')
print('='*60)


total_samples = len(train_dataset) + len(val_dataset) + len(test_dataset)
print(f'Total Images     : {total_samples}')
print(f'Train / Val / Test: {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}')
print(f'Number of Classes: 102')
print(f'Avg per class    : {total_samples / 102:.1f} images')
print()


print('Image Resolution (after resize): 224 x 224')
print('Original sizes vary — torchvision resizes to 224x224')
print()


desc_lengths = [len(CLASS_DESCRIPTIONS[i].split()) for i in range(102)]
print(f'Description Length Stats (words):')
print(f'  Min  : {min(desc_lengths)}')
print(f'  Max  : {max(desc_lengths)}')
print(f'  Mean : {np.mean(desc_lengths):.1f}')
print(f'  Std  : {np.std(desc_lengths):.1f}')
print()


train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
unique, counts = np.unique(train_labels, return_counts=True)
print(f'Train set class distribution:')
print(f'  Unique classes present: {len(unique)}')
print(f'  Min samples per class : {counts.min()}')
print(f'  Max samples per class : {counts.max()}')
print(f'  Mean samples per class: {counts.mean():.1f}')


plt.figure(figsize=(15, 4))
plt.bar(unique, counts, color='steelblue', alpha=0.8)
plt.xlabel('Class Index')
plt.ylabel('Number of Samples')
plt.title('Oxford 102 Flowers — Class Distribution in Training Set')
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 4))
plt.hist(desc_lengths, bins=15, color='coral', alpha=0.8, edgecolor='black')
plt.xlabel('Description Length (words)')
plt.ylabel('Frequency')
plt.title('Distribution of Text Description Lengths')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Visualize Images Paired With Text Descriptions
print('Visualizing image-text pairs...')


indices  = random.sample(range(len(train_dataset)), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, idx in zip(axes.flatten(), indices):
    img_tensor, label = train_dataset[idx]
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)

    flower_name = FLOWER_NAMES[label]
    description = CLASS_DESCRIPTIONS[label]

    ax.imshow(img_np)
    ax.set_title(f'Class {label}: {flower_name}', fontsize=10, fontweight='bold')
    ax.set_xlabel(description, fontsize=7, wrap=True)
    ax.axis('off')

plt.suptitle('Oxford 102 Flowers: Image-Text Pairs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Visualization complete!')

In [ ]:
# CELL 8: Color Analysis per class
print('Analyzing color statistics per class...')


class_colors = {}
for class_idx in range(0, 102, 10):
    samples = [train_dataset[i][0] for i in range(len(train_dataset))
               if train_dataset[i][1] == class_idx]
    if samples:
        stacked   = torch.stack(samples)
        mean_rgb  = stacked.mean(dim=[0, 2, 3])
        class_colors[class_idx] = mean_rgb.numpy()


fig, axes = plt.subplots(1, len(class_colors), figsize=(15, 2))
for ax, (cls_idx, color) in zip(axes, class_colors.items()):
    color_patch = np.ones((50, 50, 3)) * color
    color_patch = np.clip(color_patch, 0, 1)
    ax.imshow(color_patch)
    ax.set_title(f'{FLOWER_NAMES[cls_idx][:10]}', fontsize=6)
    ax.axis('off')
plt.suptitle('Dominant Color per Flower Class (every 10th class)', fontsize=11)
plt.tight_layout()
plt.show()
print('Color analysis done!')

In [ ]:
# CELL 9: Gradio UI — Interactive Dataset Explorer
import gradio as gr

def explore_dataset(class_name_input, split_choice, num_images):

    class_name_input = class_name_input.strip().lower()
    matched_idx = None
    for i, name in enumerate(FLOWER_NAMES):
        if class_name_input in name.lower() or name.lower() in class_name_input:
            matched_idx = i
            break

    if matched_idx is None:
        return None, f'Class not found: "{class_name_input}". Try: rose, sunflower, tulip, lotus, hibiscus'


    ds = train_dataset if split_choice == 'Train' else (val_dataset if split_choice == 'Val' else test_dataset)


    matched = [(ds[i][0], ds[i][1]) for i in range(len(ds)) if ds[i][1] == matched_idx]

    if not matched:
        return None, f'No images found for class {matched_idx} in {split_choice} split'

    num_images = min(int(num_images), len(matched))
    selected   = random.sample(matched, num_images)


    cols = min(num_images, 3)
    rows = (num_images + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if num_images == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]

    for i, (img_tensor, label) in enumerate(selected):
        r, c    = i // cols, i % cols
        img_np  = img_tensor.permute(1, 2, 0).numpy()
        img_np  = np.clip(img_np, 0, 1)
        axes[r][c].imshow(img_np)
        axes[r][c].set_title(f'Class {label}: {FLOWER_NAMES[label]}', fontsize=9)
        axes[r][c].axis('off')


    for i in range(num_images, rows * cols):
        axes[i // cols][i % cols].axis('off')

    plt.suptitle(f'Oxford 102 Flowers — {FLOWER_NAMES[matched_idx].title()} ({split_choice})', fontsize=13)
    plt.tight_layout()

    description = CLASS_DESCRIPTIONS[matched_idx]
    info = f"""Dataset Info:
Class Index  : {matched_idx}
Class Name   : {FLOWER_NAMES[matched_idx]}
Split Used   : {split_choice}
Found        : {len(matched)} images in this split
Showing      : {num_images} images

Text Description:
\"{description}\"

Description Stats:
Word count   : {len(description.split())} words
Char count   : {len(description)} characters
"""
    return fig, info


def get_dataset_stats():
    stats = f"""Oxford 102 Flowers Dataset Statistics:
{'='*45}
Total Images       : {len(train_dataset) + len(val_dataset) + len(test_dataset)}
Train / Val / Test : {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}
Number of Classes  : 102
Image Resolution   : 224 x 224 (resized)
Avg per class      : {(len(train_dataset) + len(val_dataset) + len(test_dataset)) / 102:.1f}

Description Stats (across 102 classes):
Min length         : {min(len(CLASS_DESCRIPTIONS[i].split()) for i in range(102))} words
Max length         : {max(len(CLASS_DESCRIPTIONS[i].split()) for i in range(102))} words
Mean length        : {np.mean([len(CLASS_DESCRIPTIONS[i].split()) for i in range(102)]):.1f} words

Sample class names :
{chr(10).join(f'  {i}: {FLOWER_NAMES[i]}' for i in [0, 25, 50, 75, 101])}
"""
    return stats


with gr.Blocks(title='Oxford 102 Flowers Explorer', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Oxford 102 Flowers Dataset Explorer
    ### Q4: Load, analyze and visualize a public image-text dataset
    """)

    with gr.Tab('Dataset Explorer'):
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown('### Search by Flower')
                class_input = gr.Textbox(
                    label='Flower Name',
                    placeholder='e.g. rose, sunflower, lotus, hibiscus',
                    value='rose'
                )
                split_choice = gr.Dropdown(
                    choices=['Train', 'Val', 'Test'],
                    value='Train',
                    label='Dataset Split'
                )
                num_images = gr.Slider(
                    minimum=1, maximum=6, value=3, step=1,
                    label='Number of Images'
                )
                explore_btn = gr.Button('Explore', variant='primary', size='lg')

                gr.Markdown('### Quick Picks')
                with gr.Row():
                    gr.Button('Rose').click(lambda: 'rose', outputs=class_input)
                    gr.Button('Lotus').click(lambda: 'lotus', outputs=class_input)
                    gr.Button('Sunflower').click(lambda: 'sunflower', outputs=class_input)

            with gr.Column(scale=2):
                output_plot = gr.Plot(label='Images')
                output_info = gr.Textbox(label='Class & Description Info', lines=16, interactive=False)

        explore_btn.click(
            fn=explore_dataset,
            inputs=[class_input, split_choice, num_images],
            outputs=[output_plot, output_info]
        )

    with gr.Tab('Dataset Statistics'):
        stats_btn = gr.Button('Load Statistics', variant='primary')
        stats_out = gr.Textbox(label='Full Dataset Statistics', lines=25, interactive=False)
        stats_btn.click(fn=get_dataset_stats, outputs=stats_out)

demo.launch(share=True, debug=True)